# SQL Analysis: Delhi-NCR Quick Commerce Capstone

This notebook lets you run SQL queries step by step inside Jupyter Notebook.

We will use SQLite in memory, which means you do not need to install MySQL, PostgreSQL, or SQL Server.

## 1. Import Libraries

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

## 2. Load Cleaned Dataset

This SQL notebook uses the cleaned CSV created from Python.

In [ ]:
project_root = Path(r"C:\Users\caros\OneDrive\Documents\SQL project")
data_path = project_root / "data" / "processed" / "orders_clean.csv"

orders = pd.read_csv(data_path)
orders.head()

## 3. Create SQLite Database Table

This creates a temporary SQL table named `orders_clean`.

In [ ]:
conn = sqlite3.connect(":memory:")
orders.to_sql("orders_clean", conn, index=False, if_exists="replace")

print("SQL table created successfully")

## 4. Helper Function To Run SQL

This function makes SQL output easier to read in Jupyter.

In [ ]:
def run_query(query):
    return pd.read_sql_query(query, conn)

## 5. Check Table Columns

In [ ]:
query = """
PRAGMA table_info(orders_clean);
"""

run_query(query)

## 6. Query 1: Executive KPIs

This query gives the main project KPIs.

In [ ]:
query = """
SELECT
    COUNT(*) AS total_orders,
    ROUND(SUM(order_value_inr), 2) AS total_revenue_inr,
    ROUND(AVG(order_value_inr), 2) AS avg_order_value_inr,
    ROUND(AVG(total_delivery_time_mins), 2) AS avg_delivery_time_mins,
    ROUND(AVG(sla_breached) * 100, 2) AS sla_breach_rate_pct,
    ROUND(AVG(rider_rating), 2) AS avg_rider_rating
FROM orders_clean;
"""

run_query(query)

## 7. Query 2: Store Performance

This query finds which dark stores have the highest SLA breach rate.

In [ ]:
query = """
SELECT
    dark_store_id,
    COUNT(*) AS orders,
    ROUND(SUM(order_value_inr), 2) AS revenue_inr,
    ROUND(AVG(order_value_inr), 2) AS avg_order_value_inr,
    ROUND(AVG(total_delivery_time_mins), 2) AS avg_delivery_time_mins,
    ROUND(AVG(sla_breached) * 100, 2) AS sla_breach_rate_pct,
    ROUND(AVG(delivery_distance_km), 2) AS avg_distance_km,
    ROUND(AVG(rider_rating), 2) AS avg_rider_rating
FROM orders_clean
GROUP BY dark_store_id
ORDER BY sla_breach_rate_pct DESC;
"""

run_query(query)

## 8. Query 3: Hourly Demand

This query shows which hours have the highest order volume.

In [ ]:
query = """
SELECT
    order_hour,
    COUNT(*) AS orders,
    ROUND(SUM(order_value_inr), 2) AS revenue_inr,
    ROUND(AVG(total_delivery_time_mins), 2) AS avg_delivery_time_mins,
    ROUND(AVG(sla_breached) * 100, 2) AS sla_breach_rate_pct
FROM orders_clean
GROUP BY order_hour
ORDER BY orders DESC;
"""

run_query(query).head(10)

## 9. Query 4: Root Cause Analysis

This query checks traffic, weather, and vehicle combinations that create late deliveries.

In [ ]:
query = """
SELECT
    traffic_density,
    weather,
    vehicle_type,
    COUNT(*) AS orders,
    ROUND(AVG(total_delivery_time_mins), 2) AS avg_delivery_time_mins,
    ROUND(AVG(sla_breached) * 100, 2) AS sla_breach_rate_pct,
    ROUND(AVG(rider_rating), 2) AS avg_rider_rating
FROM orders_clean
GROUP BY traffic_density, weather, vehicle_type
HAVING COUNT(*) >= 25
ORDER BY sla_breach_rate_pct DESC, orders DESC;
"""

run_query(query).head(10)

## 10. Query 5: Category Revenue Analysis

This query shows which product categories generate the most revenue.

In [ ]:
query = """
SELECT
    primary_category,
    COUNT(*) AS orders,
    ROUND(SUM(order_value_inr), 2) AS revenue_inr,
    ROUND(AVG(order_value_inr), 2) AS avg_order_value_inr,
    ROUND(AVG(sla_breached) * 100, 2) AS sla_breach_rate_pct
FROM orders_clean
GROUP BY primary_category
ORDER BY revenue_inr DESC;
"""

run_query(query)

## 11. Query 6: Distance Bucket SLA Analysis

This query checks whether longer delivery distances create more SLA breaches.

In [ ]:
query = """
SELECT
    distance_bucket,
    COUNT(*) AS orders,
    ROUND(AVG(delivery_distance_km), 2) AS avg_distance_km,
    ROUND(AVG(total_delivery_time_mins), 2) AS avg_delivery_time_mins,
    ROUND(AVG(sla_breached) * 100, 2) AS sla_breach_rate_pct
FROM orders_clean
GROUP BY distance_bucket
ORDER BY avg_distance_km;
"""

run_query(query)

## 12. Query 7: Rider Performance

This query ranks riders with at least 20 orders.

In [ ]:
query = """
SELECT
    rider_id,
    COUNT(*) AS orders,
    ROUND(AVG(total_delivery_time_mins), 2) AS avg_delivery_time_mins,
    ROUND(AVG(sla_breached) * 100, 2) AS sla_breach_rate_pct,
    ROUND(AVG(rider_rating), 2) AS avg_rider_rating,
    ROUND(AVG(delivery_speed_kmph), 2) AS avg_delivery_speed_kmph
FROM orders_clean
GROUP BY rider_id
HAVING COUNT(*) >= 20
ORDER BY sla_breach_rate_pct DESC, orders DESC;
"""

run_query(query).head(10)

## 13. Query 8: Local Rider Impact

This is one of the most important queries in the project. It compares local and non-local riders.

In [ ]:
query = """
SELECT
    is_local_rider,
    COUNT(*) AS orders,
    ROUND(SUM(order_value_inr), 2) AS revenue_inr,
    ROUND(AVG(order_value_inr), 2) AS avg_order_value_inr,
    ROUND(AVG(total_delivery_time_mins), 2) AS avg_delivery_time_mins,
    ROUND(AVG(sla_breached) * 100, 2) AS sla_breach_rate_pct,
    ROUND(AVG(delivery_distance_km), 2) AS avg_distance_km,
    ROUND(AVG(rider_rating), 2) AS avg_rider_rating
FROM orders_clean
GROUP BY is_local_rider
ORDER BY is_local_rider DESC;
"""

run_query(query)

## 14. Query 9: Local Rider Improvement Calculation

This query calculates the SLA improvement and time saving from local riders.

In [ ]:
query = """
WITH rider_type AS (
    SELECT
        is_local_rider,
        AVG(sla_breached) * 100 AS sla_breach_rate_pct,
        AVG(total_delivery_time_mins) AS avg_delivery_time_mins
    FROM orders_clean
    GROUP BY is_local_rider
)
SELECT
    ROUND(non_local.sla_breach_rate_pct - local.sla_breach_rate_pct, 2) AS sla_improvement_percentage_points,
    ROUND(non_local.avg_delivery_time_mins - local.avg_delivery_time_mins, 2) AS time_saving_minutes
FROM rider_type local
JOIN rider_type non_local
WHERE local.is_local_rider = 'Yes'
  AND non_local.is_local_rider = 'No';
"""

run_query(query)

## 15. Export SQL Query Results

This cell saves important SQL outputs as CSV files.

In [ ]:
results_dir = project_root / "sql" / "results_from_notebook"
results_dir.mkdir(parents=True, exist_ok=True)

queries = {
    "executive_kpis": """
        SELECT COUNT(*) AS total_orders,
               ROUND(SUM(order_value_inr), 2) AS total_revenue_inr,
               ROUND(AVG(total_delivery_time_mins), 2) AS avg_delivery_time_mins,
               ROUND(AVG(sla_breached) * 100, 2) AS sla_breach_rate_pct
        FROM orders_clean;
    """,
    "local_rider_impact": """
        SELECT is_local_rider,
               COUNT(*) AS orders,
               ROUND(AVG(total_delivery_time_mins), 2) AS avg_delivery_time_mins,
               ROUND(AVG(sla_breached) * 100, 2) AS sla_breach_rate_pct
        FROM orders_clean
        GROUP BY is_local_rider;
    """
}

for name, sql in queries.items():
    run_query(sql).to_csv(results_dir / f"{name}.csv", index=False)

print("SQL result files saved to:", results_dir)

## 16. Final SQL Insight

The SQL analysis confirms that local riders perform better than non-local riders.

- Local riders have lower SLA breach rate.
- Local riders have lower average delivery time.
- Business recommendation: assign local riders to high-risk zones, peak hours, bad weather, and long-distance orders.